# 15. Daily revenue with the missing days filled in

**Track:** sql | **Difficulty:** medium | **Tags:** date spine, calendar table, cross join, left join, gaps

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("015-daily-revenue-missing-days")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "calendar": """
UEFSMRUEFagLFbILTBXqAhUAEgAA1AX00wJ5TgAAek4AAHtOAAB8TgAAfU4AAH5OAAB/TgAAgE4AAIFOAACCTgAAg04AAIROAACF
TgAAhk4AAIdOAACITgAAiU4AAIpOAACLTgAAjE4AAI1OAACOTgAAj04AAJBOAACRTgAAkk4AAJNOAACUTgAAlU4AAJZOAACXTgAA
mE4AAJlOAACaTgAAm04AAJxOAACdTgAAnk4AAJ9OAACgTgAAoU4AAKJOAACjTgAApE4AAKVOAACmTgAAp04AAKhOAACpTgAAqk4A
AKtOAACsTgAArU4AAK5OAACvTgAAsE4AALFOAACyTgAAs04AALROAAC1TgAAtk4AALdOAAC4TgAAuU4AALpOAAC7TgAAvE4AAL1O
AAC+TgAAv04AAMBOAADBTgAAwk4AAMNOAADETgAAxU4AAMZOAADHTgAAyE4AAMlOAADKTgAAy04AAMxOAADNTgAAzk4AAM9OAADQ
TgAA0U4AANJOAADTTgAA1E4AANVOAADWTgAA104AANhOAADZTgAA2k4AANtOAADcTgAA3U4AAN5OAADfTgAA4E4AAOFOAADiTgAA
404AAOROAADlTgAA5k4AAOdOAADoTgAA6U4AAOpOAADrTgAA7E4AAO1OAADuTgAA704AAPBOAADxTgAA8k4AAPNOAAD0TgAA9U4A
APZOAAD3TgAA+E4AAPlOAAD6TgAA+04AAPxOAAD9TgAA/k4AAP9OAAAATwAAAU8AAAJPAAADTwAABE8AAAVPAAAGTwAAB08AAAhP
AAAJTwAACk8AAAtPAAAMTwAADU8AAA5PAAAPTwAAEE8AABFPAAASTwAAE08AABRPAAAVTwAAFk8AABdPAAAYTwAAGU8AABpPAAAb
TwAAHE8AAB1PAAAeTwAAH08AACBPAAAhTwAAIk8AACNPAAAkTwAAJU8AACZPAAAnTwAAKE8AAClPAAAqTwAAK08AACxPAAAtTwAA
FQAVggMVigMsFeoCFRAVBhUGHBgELU8AABgEeU4AABYAKAQtTwAAGAR5TgAAEREAAADBAfDAAwAAAOoCAQgvAAECAwQFBgcICQoL
DA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVW
V1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6Ch
oqOkpaanqKmqq6ytrq+wsbKztAAAABUAFTwVJiwV6gIVABUGFQYcGAEBGAEAFgAoAQEYAQAREQAAAB40AwAAAOoCARgMBoPBYDA+
BwAVBBk8NQAYBnNjaGVtYRUEABUCJQIYCGNhbF9kYXRlJQxMbAAAABUAJQIYCmlzX3dlZWtlbmQAFuoCGRwZLCYAHBUCGTUABhAZ
GAhjYWxfZGF0ZRUCFuoCFrAPFsIPJtwLJggcGAQtTwAAGAR5TgAAFgAoBC1PAAAYBHlOAAAREQAZLBUEFQAVAgAVABUQFQIAPCkG
GSYA6gIAAAAmABwVABklBgAZGAppc193ZWVrZW5kFQIW6gIWhAEWbibKDzwYAQEYAQAWACgBARgBABERABkcFQAVABUCADwpBhkm
AOoCAAAAFrQQFuoCJggWsBAAGRwYDEFSUk9XOnNjaGVtYRj4AS8vLy8vN0FBQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFB
QUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFJQUFBQklBQUFBQkFBQUFORC8vLzhBQUFFR0VBQUFBQ0FBQUFBRUFBQUFB
QUFBQUFvQUFBQnBjMTkzWldWclpXNWtBQUFFQUFRQUJBQUFBQkFBRkFBSUFBWUFCd0FNQUFBQUVBQVFBQUFBQUFBQkNCQUFBQUFn
QUFBQUJBQUFBQUFBQUFBSUFBQUFZMkZzWDJSaGRHVUFBQVlBQ0FBR0FBWUFBQUFBQUFBQUFBQUFBQT09ABggcGFycXVldC1jcHAt
YXJyb3cgdmVyc2lvbiAyNS4wLjEZLBwAABwAAAAoAgAAUEFSMQ==
""",
    "sales": """
UEFSMRUEFdA3FYYcTBX6BhUAEgAA6BsIYeoABQEAYg0IAGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABt
DQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUNCAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgA
gA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACIDQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0I
AJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgAmw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUN
CACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0IAK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4
DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMANCADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgA
yw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADTDQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0IANsNCADcDQgA3Q0I
AN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0NCADuDQgA7w0IAPAN
CADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYNCAD3DQgA+A0IAPkNCAD6DQgA+w0IAPwNCAD9DQgA/g0IAP8NCAQA64X3BAABDQgAAg0I
AAMNCAAEDQgABQ0IAAYNCAAHDQgACA0IAAkNCAAKDQgACw0IAAwNCAANDQgADg0IAA8NCAAQDQgAEQ0IABINCAATDQgAFA0IABUN
CAAWDQgAFw0IABgNCAAZDQgAGg0IABsNCAAcDQgAHQ0IAB4NCAAfDQgAIA0IACENCAAiDQgAIw0IACQNCAAlDQgAJg0IACcNCAAo
DQgAKQ0IACoNCAArDQgALA0IAC0NCAAuDQgALw0IADANCAAxDQgAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgA
Ow0IADwNCAA9DQgAPg0IAD8NCABADQgAQQ0IAEINCABDDQgARA0IAEUNCABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0I
AE4NCABPDQgAUA0IAFENCABSDQgAUw0IAFQNCABVDQgAVg0IAFcNCABYDQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGAN
CABhDQgAYg0IAGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABtDQgAbg0IAG8NCABwDQgAcQ0IAHINCABz
DQgAdA0IAHUNCAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgA
hg0IAIcNCACIDQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0I
AJkNCACaDQgAmw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsN
CACsDQgArQ0IAK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+
DQgAvw0IAMANCADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IAM8NCADQDQgA
0Q0IANINCADTDQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0IANsNCADcDQgA3Q0IAN4NCADfDQgA4A0IAOENCADiDQgA4w0I
AOQNCADlDQgA5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0NCADuDQgA7w0IAPANCADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYN
CAD3DQgA+A0IAPkNCAD6DQgA+w0IAPwNCAD9DQgA/g0IAP8NCAQA7BoACA0IAAINCAADDQgABA0IAAUNCAAGDQgABw0IAAgNCAAJ
DQgACg0IAAsNCAAMDQgADQ0IAA4NCAAPDQgAEA0IABENCAASDQgAEw0IABQNCAAVDQgAFg0IABcNCAAYDQgAGQ0IABoNCAAbDQg8
HOwAAAAAAAAd7AAAAAAAABUAFYIIFYwILBX6BhUQFQYVBhwYCB3sAAAAAAAAGAhh6gAAAAAAABYAKAgd7AAAAAAAABgIYeoAAAAA
AAAREQAAAIEE9AACAwAAAPoGAQlxAAIIGECggIEDCBIoWMCggYMHECJImEChgoULGDJo2MChg4cPIEKIGEGihIkTKFKoWMGihYsX
MGLImEGjho0bOHLo2MGjh48fQIIIGUKkiJEjSJIoWcKkiZMnUKJImUKlipUrWLJo2cKli5cvYMKIGUOmjJkzaNKoWcOmjZs3cOLI
mUOnjp07ePLo2cOnj58/gAIJGkSokKFDiBIpWsSokaNHkCJJmkSpkqVLmDJp2sSpk6dPoEKJGkWqlKlTqFKpWsWqlatXsGLJmkWr
lq1buHLp2sWrl69fwIIJG0asmLFjyJIpW8asmbNn0KJJm0atmrVr2LJp28atm7dv4MKJG0eunLlz6NKpW8eunbt38OLJm0evnr17
+PLp28evn79/AAMKHEiwoMGDCBMqXMiwocOHECNKnEixosWLGDNq3Mixo8ePIEOKHEmypMmTKFOqXMmypcuXMGPKnEmzps2bOHPq
3Mmzp8+fQIMKHUq0qNGjSJMqXcq0qdOnUKNKnUq1qtWrWLNq3cq1q9evYMOKHUu2rNmzaNOqXcu2rdu3cOPKnUu3rt27ePPq3cu3
r9+/gAMLHky4sOHDiBMrXsy4sePHkCNLnky5suXLmDNr3sy5s+fPoEOLHk26tOnTqFOrXs26tevXsGPLnk27tu3buHPr3s0bAAAA
FQQVYBVITBUMFQASAAAwBAQACQEAAwkHBAACDQgABQ0IPAYAAAAAAAAAAQAAAAAAAAAVABXiAhXqAiwV+gYVEBUGFQYcGAgGAAAA
AAAAABgIAQAAAAAAAAAWACgIBgAAAAAAAAAYCAEAAAAAAAAAEREAAACxAfCwAwAAAPoGAQNxiAIJ2MaVbQpVGpNmEJR2yAIFUzNs
01KJaaEowia1SdABDYs2TRIwqgJhkMKUWlKkBCAmqRJsUqWtAVohXVsRastFikpgUzpQSlpASTsAA7atmyIgGKZgAgRyWLgqCAQK
AMaQXIZpQMaUULpJgAKgQREEWrqUANCgChE0kQAUwBqAbKM0BCS1BTkNKhWsqdgoTNNMjRBxbbEiQCVQCpBpCBYRZASFmxAAFQQV
2AMV4ANMFXYVABIAAOwB8OvMTgAA0U4AALVOAAC5TgAAw04AAK5OAADHTgAA2U4AALxOAAC7TgAAv04AAK9OAAC0TgAA004AAK1O
AADGTgAAwU4AAMhOAADUTgAAuk4AAOBOAACxTgAAqU4AAL5OAADOTgAArE4AALZOAACwTgAApk4AAL1OAADNTgAA204AAKpOAADL
TgAAxE4AAN1OAACoTgAAz04AAMlOAADaTgAA1U4AAKtOAADcTgAA0k4AANZOAADKTgAAsk4AAN5OAADhTgAA104AALhOAACnTgAA
304AAMVOAADQTgAAwk4AALNOAADATgAAt04AABUAFbIFFbwFLBX6BhUQFQYVBhwYBOFOAAAYBKZOAAAWACgE4U4AABgEpk4AABER
AAAA2QL0WAEDAAAA+gYBBnFAIAxEYRxIoiwF0zgPFEkTVVkXlmlRtHGdBwjHB2KcFjKikXXBSI6bkW0lpmWmcFoKCaKpKipSxSQn
NKwHe7BT6y4Q9LrUuaoaDMYdaLkyOsYz3Q5dHdrkijaJUT/bDV+vVIYSnm+exBppJRMjzciLDopE4l2eNDrrqNIJdVZoBE7cIxSQ
8eUzqwFc2EJU+eVJIKAMgZmBk6eDAC+DeAvEbeDGbS5DZr9GAFDRDUbpPCyi2qjDsX6KENWxHZ46iuAuG4Ushh7cq20edOmYhT3q
phW0CAqWNSMpN3RQXNJyaNNKc1No1GpIu1mBlifKPYlXw75QwMUgGLRyXCAYzlywdLiqHmc2ubK4DaXmsRFkA3+JJZOw4IADrZhM
1cKEaRu4NzRV/rbRB8ovvQUEc8pIeT/ZMaCW7dwLmjDRC5+c2lY1dMxwsQRI+XIAAAcjAAAVBBWwIhW6IkwV8AYVABIAAJgR9JcI
AAAGey4AAAyAAAAAA5YqAAAEUtYAAAXqdAAACCIIAAAFzSgAAAht0AAACcKiAAAFmpIAAAKPRgAAByjQAAAHzK4AAAMR8AAACVXY
AAAIIaQAAA2LlAAABtAkAAAHlWgAAAvnRAAACIoiAAALet4AAAlFDAAABN+oAAAJdXwAAAtHsgAABKhiAAAGaQQAAAx/zgAABYYQ
AAAKFBQAAAou1gAADWDOAAANOfAAAAe1PgAAB8qIAAAFgS4AAARi2gAABTMOAAAHDqQAAA26pgAAATwEAAAJesIAAArEcgAAAr3C
AAAH27gAAAwm8AAACIc0AAAMZNoAAA0togAAAmcKAAAG15AAAAZ+gAAAA948AAAFoKAAAAOF9AAACNvGAAAIe64AAAG/2gAAALfK
AAAJkKIAAAe2nAAADJZ2AAAGoHwAAANBNAAAApI0AAAFvVYAAAXeigAABr9YAAAL+jYAAAQ1WAAACX4UAAAISOYAAAyxBgAADCJy
AAAKktYAAAZT7AAACmNgAAALsuwAAAFSFgAAAkIgAAACW4QAAAK1XAAABtbIAAAKNqYAAAKFtAAADFiMAAANF/QAAAzrbAAACx20
AAAMYrQAAALA4gAABakGAAAAmLwAAAZN3gAAB26KAAABB94AAANWfgAABZO8AAAF3u4AAAIg7AAACI+aAAACzGgAAAa7ogAAB0W4
AAACsEgAAAF7sAAAC33+AAAGfPAAAAtgHAAAA844AAANfhoAAAs8LAAAC/60AAAJ29QAAAcWEAAACPa6AAABg04AAAL10AAAAlYM
AAABVKAAAADiwgAAAPJiAAAGr+oAAAmw3AAADG1AAAAHcaoAAA11HgAABeQ0AAAHm3YAAAW0KAAAB2s4AAAFkl4AAAweWAAADHu0
AAANlPQAAAM3cAAACtFWAAAFNNAAAAZDUgAACrt2AAAG55QAAAPv0AAAAtF8AAADQ1oAAAzZQgAABEtqAAAIzLwAAAK6cAAAC1FE
AAADYHQAAAZAAAAABzOOAAAC5QQAAAHRCgAABbv4AAALRiIAAAKoeAAADG80AAADiUYAAAgOTgAACkEAAAAHXC4AAAFnxAAABmjS
AAAI+XYAAAGErAAABHiIAAANa74AAAcurAAABbYcAAAFxMIAAAXiDgAACPoMAAAAdioAAAYlcAAADHEoAAAC5NIAAAMEdgAADP1k
AAAKF5gAAAwIeAAABB5MAAAJMegAAAsjLAAAC5KyAAAHsFwAAAlGOAAADKWyAAAJpVYAAAkscAAACJD4AAAAiOoAAAC2bAAACkta
AAAJHAgAAAsT8AAAB5wMAAAGwhQAAAeIIAAADa8gAAALGQQAAArLegAAC5JOAAAMsD4AAAlLsAAAC8duAAAIW6YAAARjPgAACbrS
AAABUCIAAAQZBgAACRGuAAAHD54AAARI4AAAAPPAAAABg7IAAADrKAAACKH2AAAGgaAAAAuZJAAAATa+AAACAzwAAA2CAgAAChvk
AAAHx8wAAAslhAAAAfbuAAAL2coAAAjtWgAAAGGoAAAIFSQAAAKC+AAAA/GSAAAM4EoAAA1jWAAAAtDmAAALImQAAAexVgAAA3Va
AAAIvxAAAAoGzAAAAoIwAAAEUH4AAAhvkgAACbQuAAAGEuIAAACU1AAAAHaOAAAFFu4AAAnRSAAACVOyAAAK67QAAAKPFAAAAPL4
AAAJXzgAAAR6fAAACMA8AAAKcs4AAAK99AAADVSAAAAIj8wAAAf8VgAAAxvmAAAAvKwAAAL6HAAACcNqAAABumIAAAr4/AAAAu1q
AAAKH5oAAAsgPgAADVg2AAADGfIAAArQjgAAAlPmAAAFuEIAAAzpeAAACrSgAAAGbq4AAAiAXgAAAMLsAAAChOwAAAgRoAAABETG
AAAIeVYAAAE53gAADN0qAAAKJOAAAAixMgAACYPwAAAF+3IAAAxlDAAABa22AAACE3IAAA0U1AAADHZuAAAI6KoAAARXhgAABvtO
AAABDCoAAAKQQAAAA8+WAAAI0soAAAGgaAAACqC0AAAIf5YAAAu21AAAA+uEAAAKncYAAAIZTgAABjLqAAADF5oAAAbwXgAAC5za
AAAGNwQAAAP0gAAACj8+AAABiY4AAAnx5gAAC7jIAAAKnZQAAAcawAAAB6mGAAAKmd4AAAYb3gAACmFsAAALwfYAAAciwgAABsiG
AAACjCYAAAEnUAAAAG4oAAAAkkoAAAlHZAAACcT6AAAH1OIAAAMbggAAAc8WAAAM57YAAACvMgAABYDKAAAKxpgAAAkF9gAAA2kM
AAADsCQAAApnrAAAAzCaAAABQUoAAAs/sAAAAjx2AAANTH4AAAo4mgAAC50+AAAG1ywAAA18vAAAAQW4AAANgA4AAA245AAAAlj6
AAALhHAAAAzsmAAAA2I2AAAL/boAAAIqfgAACzkMAAADOcgAAARsbAAACpmsAAAMFcAAAA0q5gAACbT2AAAF7P4AAAsXpgAACaWI
AAAIVQIAAAfiKgAACknKAAAFAUAAAATvegAABQRgAAAGZOoAAAMUrAAAAfeEAAAAi6YAAAqr1gAAAS8gAAAEW24AAABLlgAAATjk
AAADpAgAAAbemAAACClCAAAJco4AAAFdOAAAALc0AAAEXioAAAKtKAAACLnKAAAICsoAAAHp2AAAAfB8AAAHOD4AAAo9fAAAB8jG
AAAG2YQAAAdSzgAABCoEAAADpZgAAAtzcgAAColEAAANqzgAAAQFTAAABhFSAAALw7gAAACowAAAAL/+AAAMV8QAAAy8WgAACywo
AAABCzAAAAY/zgAACFJ4AAAIWqwAAALMmgAABocYAAAAvzYAAAilrAAAAaIqAAAEWwoAAASUdgAABAg6AAAHvKoAAAXZRAAAAjHq
AAACWfQAAAF49AAABxg2AAADg5wAAAICEBUAFYIIFYwILBX6BhUQFQYVBhwYBQAADbqmGAUAAABLlhYAKAUAAA26phgFAAAAS5YR
EQAAAIEE9AACAwAAAPoGAQlxAAIIGECggIEDCBIoWMCggYMHECJImEChgoULGDJo2MChg4cPIEKIGEGihIkTKFKoWMGihYsXMGLI
mEGjho0bOHLo2MGjh48fQIIIGUKkiJEjSJIoWcKkiZMnUKJImUKlipUrWLJo2cKli5cvYMKIGUOmjJkzaNKoWcOmjZs3cOLImUOn
jp07ePLo2cOnj58/gAIJGkSokKFDiBIpWsSokaNHkCJJmkSpkqVLmJ5k0rSJUydPn0CFEjWKVClTp1ClUrWKVStXr2DFkjWLVi1b
t3Dl0rWLVy9fv4AFEzaMWDFjx5AlU7aMWTNnz6BFkzaNWjVr17Bl07aNWzdv38CFEzeOXDlz59ClU3dtHbt27t7BiydvHr169u7h
y6dvH79+/v4BDChwIMGCBg8iTKhwIcOGDh9CjChxIsWKFi9i9JZR40aOHT1+BBlS5EiSJU2eRJlS5UqWLV2+hBlT5kyagWravIkz
p86dPHv6/Ak0qNChRIsaPYo0qdKlTJs6fQo1qtSpVKtavYo1q9atXLt6/Qo2rNixZMuaPYs2rdq1bA+1dfsWbly5c+nWtXsXb169
e/n29fsXcGDBgwkXNnwYcWLFixk3dvwYcmTJkylXtnwZc2bNmzl39vwZdGjRo0mXNn0adWrVq1m3dv0admzZs2nXtn0bAAAAFQQZ
XDUAGAZzY2hlbWEVCAAVBCUCGAdzYWxlX2lkABUEJQIYCHN0b3JlX2lkABUCJQIYCXNhbGVfZGF0ZSUMTGwAAAAVDhUKFQIYBmFt
b3VudCUKFQQVFCxcFQQVFAAAABb6BhkcGUwmABwVBBk1AAYQGRgHc2FsZV9pZBUCFvoGFvhAFrglJrAcJggcGAgd7AAAAAAAABgI
YeoAAAAAAAAWACgIHewAAAAAAAAYCGHqAAAAAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAPoGAAAAJgAcFQQZNQAGEBkYCHN0
b3JlX2lkFQIW+gYW4gQW0gQmpCYmwCUcGAgGAAAAAAAAABgIAQAAAAAAAAAWACgIBgAAAAAAAAAYCAEAAAAAAAAAEREAGSwVBBUA
FQIAFQAVEBUCADwpBhkmAPoGAAAAJgAcFQIZNQAGEBkYCXNhbGVfZGF0ZRUCFvoGFo4KFqAKJpIuJpIqHBgE4U4AABgEpk4AABYA
KAThTgAAGASmTgAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAPoGAAAAJgAcFQ4ZNQAGEBkYBmFtb3VudBUCFvoGFsArFtQrJo5X
JrI0HBgFAAANuqYYBQAAAEuWFgAoBQAADbqmGAUAAABLlhERABksFQQVABUCABUAFRAVAgA8KQYZJgD6BgAAABaoexb6BiYIFv5f
ABkcGAxBUlJPVzpzY2hlbWEYmAMvLy8vL3lnQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFB
QUJBQUlBQUFBQkFBQUFBUUFBQURBQUFBQWVBQUFBRUFBQUFBRUFBQUFZUC8vL3dBQUFRY1FBQUFBSUFBQUFBUUFBQUFBQUFBQUJn
QUFBR0Z0YjNWdWRBQUFDQUFNQUFRQUNBQUlBQUFBQ2dBQUFBSUFBQUNZLy8vL0FBQUJDQkFBQUFBZ0FBQUFCQUFBQUFBQUFBQUpB
QUFBYzJGc1pWOWtZWFJsQUFZQUNBQUdBQVlBQUFBQUFBQUF6UC8vL3dBQUFRSVFBQUFBSEFBQUFBUUFBQUFBQUFBQUNBQUFBSE4w
YjNKbFgybGtBQUFBQU1ELy8vOEFBQUFCUUFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQWhBQUFBQWdBQUFBQkFB
QUFBQUFBQUFIQUFBQWMyRnNaVjlwWkFBSUFBd0FDQUFIQUFnQUFBQUFBQUFCUUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2
ZXJzaW9uIDI1LjAuMRlMHAAAHAAAHAAAHAAAAO4DAABQQVIx
""",
    "stores": """
UEFSMRUEFWAVSEwVDBUAEgAAMAQBAAkBAAIJBwQAAw0IAAQNCDwFAAAAAAAAAAYAAAAAAAAAFQAVFhUaLBUMFRAVBhUGHBgIBgAA
AAAAAAAYCAEAAAAAAAAAFgAoCAYAAAAAAAAAGAgBAAAAAAAAABERAAAACygCAAAADAEDA4jGAhUEFd4BFeQBTBUMFQASAABv8G4J
AAAAUHVuZSBDYW1wFQAAAEJlbmdhbHVydSBJbmRpcmFuYWdhchQAAABIeWRlcmFiYWQgR2FjaGlib3dsaQ0AAABDaGVubmFpIEFk
eWFyDQAAAE11bWJhaSBCYW5kcmELAAAARGVsaGkgU2FrZXQVABUWFRosFQwVEBUGFQYcNgAoCVB1bmUgQ2FtcBgVQmVuZ2FsdXJ1
IEluZGlyYW5hZ2FyEREAAAALKAIAAAAMAQMDiMYCFQQZPDUAGAZzY2hlbWEVBAAVBCUCGAhzdG9yZV9pZAAVDCUCGApzdG9yZV9u
YW1lJQBMHAAAABYMGRwZLCYAHBUEGTUABhAZGAhzdG9yZV9pZBUCFgwWkAIW/AEmbCYIHBgIBgAAAAAAAAAYCAEAAAAAAAAAFgAo
CAYAAAAAAAAAGAgBAAAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgAMAAAAJgAcFQwZNQAGEBkYCnN0b3JlX25hbWUVAhYM
FoYDFpADJogEJoQCHDYAKAlQdW5lIENhbXAYFUJlbmdhbHVydSBJbmRpcmFuYWdhchERABksFQQVABUCABUAFRAVAgA8Fq4BGQYZ
JgAMAAAAFpYFFgwmCBaMBQAZHBgMQVJST1c6c2NoZW1hGIACLy8vLy83Z0FBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFB
QkJBQU1BQUFBQ0FBSUFBQUFCQUFJQUFBQUJBQUFBQUlBQUFCSUFBQUFCQUFBQU5ELy8vOEFBQUVGRUFBQUFDQUFBQUFFQUFBQUFB
QUFBQW9BQUFCemRHOXlaVjl1WVcxbEFBQUVBQVFBQkFBQUFCQUFGQUFJQUFZQUJ3QU1BQUFBRUFBUUFBQUFBQUFCQWhBQUFBQWtB
QUFBQkFBQUFBQUFBQUFJQUFBQWMzUnZjbVZmYVdRQUFBQUFDQUFNQUFnQUJ3QUlBQUFBQUFBQUFVQUFBQUFBQUFBQQAYIHBhcnF1
ZXQtY3BwLWFycm93IHZlcnNpb24gMjUuMC4xGSwcAAAcAAAAXgIAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "015-daily-revenue-missing-days"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

calendar = spark.read.parquet((_data_dir / "calendar.parquet").resolve().as_posix())
calendar.createOrReplaceTempView("calendar")
sales = spark.read.parquet((_data_dir / "sales.parquet").resolve().as_posix())
sales.createOrReplaceTempView("sales")
stores = spark.read.parquet((_data_dir / "stores.parquet").resolve().as_posix())
stores.createOrReplaceTempView("stores")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'revenue': 'df3f9ff88b05e878b9ecbfe4c1c99992c1b39508e09519ac15dfc92c3fd1562b',
                         'sale_date': '90c5b2a756ccc0e24a39ba08af32987465f9f0c90a647dae7ebb3c277e391c63',
                         'store_id': 'd7deee051a1219aa665655ef9d8134b2709eb606e7ea1097337104b769eb96ab'},
 'columns': [['revenue', 'decimal(12,2)'], ['sale_date', 'date'], ['store_id', 'bigint']],
 'fingerprint': '28ed07724ec456c6412446ba80267df0936e86d2a00bce370f99763bf9e96d28',
 'order_matters': False,
 'row_count': 186}

check = make_check(EXPECTED)

## The situation

A dashboard draws daily revenue per store as a line chart. The chart library connects
the points it is given. When a store has no sales on a day, there is no point for that
day, and the line runs straight from the day before to the day after. A closed store
looks like a normal day.

The chart needs a row for every store and every day, with 0 where nothing was sold.

## Input tables

### `stores`

One row per store.

| Column | Type | Description |
|---|---|---|
| `store_id` | BIGINT | Unique per store. |
| `store_name` | STRING | Display name. |

### `calendar`

One row per calendar day, from 1 January 2025 to 30 June 2025.

| Column | Type | Description |
|---|---|---|
| `cal_date` | DATE | The day. |
| `is_weekend` | BOOLEAN | True on Saturdays and Sundays. |

### `sales`

One row per sale. A store can have several sales on a day, or none.

| Column | Type | Description |
|---|---|---|
| `sale_id` | BIGINT | Unique per sale. |
| `store_id` | BIGINT | The store. |
| `sale_date` | DATE | Date of the sale. |
| `amount` | DECIMAL(10,2) | Sale value in rupees. |

## Requirement

Return daily revenue for March 2025.

- One row for every store in `stores` and every day from 1 March 2025 to
  31 March 2025, both included. That is 31 rows per store.
- `revenue` is the sum of the store's sales on that day, and 0.00 if it had none.
- A store with no sales in March at all still gets its 31 rows.
- Sales outside March do not appear.

## Expected output

One row per store per day of March.

| Column | Type | Description |
|---|---|---|
| `store_id` | BIGINT | |
| `sale_date` | DATE | A day of March 2025. |
| `revenue` | DECIMAL(12,2) | 0.00 on days without sales. |

In [ ]:
calendar.show(5)
sales.show(5)
stores.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `calendar`, `sales`, `stores`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)